# 1. Profiling — Raw dataset

Profile the raw eye disease dataset (cataract, diabetic_retinopathy, glaucoma, normal) using **ydata-profiling**. Output is saved to `art/report/profiling/` and displayed inline.

In [1]:
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
from ydata_profiling import ProfileReport
from cleanvision import Imagelab

In [2]:
# Same paths as eye_disease_v1: from nbs/ use ../art
ART_ROOT = os.path.join("..", "art")
if not os.path.isdir(ART_ROOT):
    raise FileNotFoundError(
        f"Artifacts folder not found: {os.path.abspath(ART_ROOT)}\n"
        "Create it at project root (e.g. art/data/raw/train, art/data/raw/validate)."
    )

raw_train_dir = os.path.join(ART_ROOT, "data", "raw", "train")
raw_val_dir = os.path.join(ART_ROOT, "data", "raw", "validate")

# Output for this notebook
PROFILING_OUT = os.path.join(ART_ROOT, "report", "profiling")
os.makedirs(PROFILING_OUT, exist_ok=True)

Build a table of image paths, labels, and **image metadata** (dimensions, resolution, PPI, file size, format, brightness, blur) for profiling. Classes: cataract, diabetic_retinopathy, glaucoma, normal.

In [3]:
def get_image_meta(path):
    """Return width, height, file_size_bytes, format, aspect_ratio, mode, brightness, blur. On error return NaNs."""
    try:
        size_bytes = os.path.getsize(path)
        with Image.open(path) as im:
            w, h = im.size
            mode = im.mode
            fmt = im.format or os.path.splitext(path)[1].lstrip(".").upper()
            gray = np.array(im.convert("L"))
            brightness = float(np.mean(gray))
            blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        return w, h, size_bytes, fmt, (w / h if h else None), mode, brightness, blur
    except Exception:
        return None, None, None, None, None, None, None, None


def collect_image_manifest(train_dir, val_dir, with_image_meta=True):
    rows = []
    for split, base_dir in [("train", train_dir), ("validate", val_dir)]:
        if not os.path.isdir(base_dir):
            continue
        for class_name in os.listdir(base_dir):
            class_path = os.path.join(base_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
            for f in (tqdm(files, desc=f"{split}/{class_name}", leave=False) if with_image_meta else files):
                path = os.path.join(class_path, f)
                if not os.path.isfile(path):
                    continue
                row = {"filepath": path, "class": class_name, "split": split}
                if with_image_meta:
                    w, h, size_b, fmt, ar, mode, brightness, blur = get_image_meta(path)
                    row["width"] = w
                    row["height"] = h
                    row["resolution"] = f"{w}x{h}" if (w is not None and h is not None) else None
                    row["pixel_count"] = w * h if (w is not None and h is not None) else None
                    row["file_size_kb"] = round(size_b / 1024, 2) if size_b is not None else None
                    row["format"] = fmt
                    row["aspect_ratio"] = round(ar, 4) if ar is not None else None
                    row["mode"] = mode
                    row["brightness"] = round(brightness, 2) if brightness is not None else None
                    row["blur"] = round(blur, 2) if blur is not None else None
                rows.append(row)
    return pd.DataFrame(rows)


df = collect_image_manifest(raw_train_dir, raw_val_dir, with_image_meta=True)
df["filepath"] = df["filepath"].apply(lambda p: os.path.normpath(p))

# CleanVision quality audit — compute issue_count per image
CV_ISSUE_COLS = [
    "is_odd_aspect_ratio_issue", "is_low_information_issue",
    "is_light_issue", "is_dark_issue", "is_blurry_issue",
    "is_grayscale_issue", "is_exact_duplicates_issue", "is_near_duplicates_issue",
]

for split, base_dir in [("train", raw_train_dir), ("validate", raw_val_dir)]:
    lab = Imagelab(data_path=base_dir)
    lab.find_issues()
    print(f"\n=== CleanVision issue summary ({split}) ===")
    print(lab.issue_summary.to_string(index=False))
    issues = lab.issues.copy()
    issues["filepath"] = issues.index.map(os.path.normpath)
    present_cols = [c for c in CV_ISSUE_COLS if c in issues.columns]
    issues["issue_count"] = issues[present_cols].sum(axis=1).astype(int)
    df = df.merge(issues[["filepath", "issue_count"]], on="filepath", how="left", suffixes=("", f"_{split}"))
    dup = f"issue_count_{split}"
    if dup in df.columns:
        df["issue_count"] = df["issue_count"].fillna(df[dup])
        df.drop(columns=[dup], inplace=True)

df["issue_count"] = df["issue_count"].fillna(0).astype(int)

df_train = df[df["split"] == "train"].copy()
df_validate = df[df["split"] == "validate"].copy()

# Force dtypes so ydata-profiling shows full stats
for _df in (df_train, df_validate):
    _df["aspect_ratio"] = pd.to_numeric(_df["aspect_ratio"], errors="coerce")
    _df["brightness"] = pd.to_numeric(_df["brightness"], errors="coerce")
    _df["blur"] = pd.to_numeric(_df["blur"], errors="coerce")
    _df["pixel_count"] = pd.to_numeric(_df["pixel_count"], errors="coerce")
    _df["mode"] = _df["mode"].astype("category")
    _df["resolution"] = _df["resolution"].astype("category")

print(f"\nTotal images: {len(df)} (train: {len(df_train)}, validate: {len(df_validate)})")
print(df_train[["class", "width", "height", "resolution", "pixel_count", "file_size_kb", "format", "mode", "brightness", "blur", "issue_count"]].head(10))
df_train.head(10)

Reading images from C:/Users/ramkan/workdrive/workspace/worklearn/bits_wilp/repo/local/edir/nbs/../art/raw/train
Checking for dark, light, odd_aspect_ratio, low_information, exact_duplicates, near_duplicates, blurry, grayscale, odd_size images ...


Computing scores:   0%|          | 0/4145 [00:00<?, ?it/s]

Computing hashes:   0%|          | 0/4145 [00:00<?, ?it/s]

Issue checks completed. 368 issues found in the dataset. To see a detailed report of issues found, use imagelab.report().

=== CleanVision issue summary (train) ===
      issue_type  num_images
        odd_size         195
            dark          91
 near_duplicates          54
          blurry          22
exact_duplicates           6
 low_information           0
odd_aspect_ratio           0
           light           0
       grayscale           0
Reading images from C:/Users/ramkan/workdrive/workspace/worklearn/bits_wilp/repo/local/edir/nbs/../art/raw/validate
Checking for dark, light, odd_aspect_ratio, low_information, exact_duplicates, near_duplicates, blurry, grayscale, odd_size images ...


Computing scores:   0%|          | 0/73 [00:00<?, ?it/s]

Computing hashes:   0%|          | 0/73 [00:00<?, ?it/s]

Issue checks completed. 8 issues found in the dataset. To see a detailed report of issues found, use imagelab.report().

=== CleanVision issue summary (validate) ===
      issue_type  num_images
        odd_size           6
            dark           1
          blurry           1
odd_aspect_ratio           0
           light           0
 low_information           0
       grayscale           0
exact_duplicates           0
 near_duplicates           0

Total images: 4218 (train: 4145, validate: 73)
      class  width  height resolution  pixel_count  file_size_kb format mode  \
0  cataract    512     512    512x512       262144         41.59   JPEG  RGB   
1  cataract    512     512    512x512       262144         45.64   JPEG  RGB   
2  cataract    512     512    512x512       262144         50.81   JPEG  RGB   
3  cataract    512     512    512x512       262144         40.07   JPEG  RGB   
4  cataract    512     512    512x512       262144         63.49   JPEG  RGB   
5  cataract    5

,filepath,class,split,width,height,resolution,pixel_count,file_size_kb,format,aspect_ratio,mode,brightness,blur,issue_count
0,..\art\raw\train\cataract\0_left.jpg,cataract,train,512,512,512x512,262144,41.59,JPEG,1.0,RGB,77.84,258.66,0
1,..\art\raw\train\cataract\103_left.jpg,cataract,train,512,512,512x512,262144,45.64,JPEG,1.0,RGB,106.13,368.18,0
2,..\art\raw\train\cataract\1062_right.jpg,cataract,train,512,512,512x512,262144,50.81,JPEG,1.0,RGB,140.32,479.90,0
3,..\art\raw\train\cataract\1083_left.jpg,cataract,train,512,512,512x512,262144,40.07,JPEG,1.0,RGB,64.86,134.30,0
4,..\art\raw\train\cataract\1084_right.jpg,cataract,train,512,512,512x512,262144,63.49,JPEG,1.0,RGB,101.04,408.10,0
5,..\art\raw\train\cataract\1102_left.jpg,cataract,train,512,512,512x512,262144,51.16,JPEG,1.0,RGB,108.04,356.18,0
6,..\art\raw\train\cataract\1102_right.jpg,cataract,train,512,512,512x512,262144,46.64,JPEG,1.0,RGB,107.51,308.06,0
7,..\art\raw\train\cataract\1115_left.jpg,cataract,train,512,512,512x512,262144,42.36,JPEG,1.0,RGB,112.57,634.09,0
8,..\art\raw\train\cataract\1126_right.jpg,cataract,train,512,512,512x512,262144,46.81,JPEG,1.0,RGB,154.00,715.64,0
9,..\art\raw\train\cataract\112_right.jpg,cataract,train,512,512,512x512,262144,38.52,JPEG,1.0,RGB,98.95,303.06,0


Run ydata-profiling **separately** for train and validate. Reports are saved to `art/report/profiling/` and shown inline.

In [4]:
# Train report
report_train = ProfileReport(
    df_train,
    title="Raw train — Eye disease (cataract, diabetic_retinopathy, glaucoma, normal)",
    minimal=False,
)
out_train = os.path.join(PROFILING_OUT, "report_train.html")
report_train.to_file(out_train)
print(f"Saved: {out_train}")
report_train.to_notebook_iframe()

# Validate report
report_validate = ProfileReport(
    df_validate,
    title="Raw validate — Eye disease (cataract, diabetic_retinopathy, glaucoma, normal)",
    minimal=False,
)
out_validate = os.path.join(PROFILING_OUT, "report_validate.html")
report_validate.to_file(out_validate)
print(f"Saved: {out_validate}")
report_validate.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 102.36it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

...


100%|███████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 145.56it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: ..\art\profiling\report_validate.html
